In [1]:
import pandas as pd
import plotly.express as px

In [2]:
extracted_data = pd.read_csv("data_files/data_1.csv")
project_metadata= pd.read_csv("data_files/project_metadata.csv")
print("Project Metadata Columns:", project_metadata.columns.tolist())
print("\nExtracted Data Columns:", extracted_data.columns.tolist())

Project Metadata Columns: ['No.', 'Country', 'Project ID', 'Project Name', 'Type', 'SE', 'Region', 'Call For Proposal', 'Status', 'SE Approval Date', 'First disbursement date of GAFSP funds', 'Additional Financing Approval Date 1', 'Project Closing Expected Date', 'Project Closing Date', 'Additonal Funding Request Amount', 'Revised Funding Request Amount', 'Remark', 'No. of PADs', 'Link']

Extracted Data Columns: ['source_file', 'PROJ_DEV_OBJECTIVE_DESC', 'LEAD_GP_NAME', 'CMT_AMT', 'Climate Financing (%)', 'Adaptation (%)', 'Mitigation (%)', 'PriorActions', 'Indicators', 'Components', 'DLI_DLR']


In [3]:
extracted_data['Project ID'] = extracted_data['source_file'].str.replace('.pdf', '', regex=False)
extracted_data = extracted_data.drop('source_file', axis=1)

In [4]:
full_data = extracted_data.merge(project_metadata, on='Project ID', how='left')

In [5]:
# Create a copy of PROJ_DEV_OBJECTIVE_DESC in a new column
full_data['PROJ_OBJECTIVE_TEXT'] = full_data['PROJ_DEV_OBJECTIVE_DESC']

# Convert SE Approval Date to year (YYYY) format
full_data['SE Approval Date'] = pd.to_datetime(full_data['SE Approval Date'], errors='coerce').dt.year

# Rename the columns
full_data = full_data.rename(columns={
    "Project ID": "PROJ_ID",
    "Project Name": "PROJ_DISPLAY_NAME",
    "SE Approval Date": "PROJ_APPRVL_FY",
    "Status": "PROJ_STAT_NAME",
    "Country": "CNTRY_SHORT_NAME",
    "Type": "LNDNG_INSTR_LONG_NAME"
})


In [6]:
# Reorder columns with specified columns at the beginning
column_order = ["PROJ_ID", "PROJ_DISPLAY_NAME", "PROJ_APPRVL_FY", "PROJ_DEV_OBJECTIVE_DESC", 
                "PROJ_STAT_NAME", "CNTRY_SHORT_NAME", "LNDNG_INSTR_LONG_NAME", "LEAD_GP_NAME", 
                "CMT_AMT", "PROJ_OBJECTIVE_TEXT", "Region", "Climate Financing (%)", 
                "Adaptation (%)", "Mitigation (%)", "PriorActions", "Indicators", "Components", "DLI_DLR"]

# Get remaining columns not in the specified order
remaining_columns = [col for col in full_data.columns if col not in column_order]

# Combine and reorder
full_data = full_data[column_order + remaining_columns]


In [7]:
hierarchy = pd.DataFrame([
    {"Hierarchy Name": "School", "Full Name": "School", "Keyword": "school"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "school feeding"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "school meals"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "students"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "pupils"},
    {"Hierarchy Name": "WFP", "Full Name": "World food programme", "Keyword": "school meals"},
    {"Hierarchy Name": "WFP", "Full Name": "World food programme", "Keyword": "school feeding"}
])



In [15]:
hierarchy = pd.DataFrame([
    {"Hierarchy Name": "School", "Full Name": "School", "Keyword": "school"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "school feeding"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "school meals"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "students"},
    {"Hierarchy Name": "HGSF", "Full Name": "home-grown school feeding", "Keyword": "pupils"},
    {"Hierarchy Name": "WFP", "Full Name": "World food programme", "Keyword": "school meals"},
    {"Hierarchy Name": "WFP", "Full Name": "World food programme", "Keyword": "school feeding"}
])
hierarchy_pim = {'Key Terms': ['public investment management'], 
                  'Additional': ['public investment management assessment', 'climate public investment management']}

hierarchy_hgsf = {'Key Terms': ['school feeding', 'school meals'], 
                   'Additional': ['students', 'pupils']}

hierarchy = {
    'School': {'Key Terms': ['school'], 'Additional': []},
    'HGSF': hierarchy_hgsf,
    'PIM': hierarchy_pim,
    'WFP': {'Key Terms': ['world food programme'], 'Additional': ['school feeding', 'school meals']}
}

In [8]:
fig1_df = full_data.groupby(['PROJ_APPRVL_FY','LNDNG_INSTR_LONG_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'LNDNG_INSTR_LONG_NAME':'Lending Instrument'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Lending Instrument',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.G10)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)

In [18]:
df_pdo_search = full_data[['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']]

#This is the function to change to incorporate new ML based solution for better categorization
def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return None
  else:
    return None

for each_group in hierarchy:
  print(f"Processing category: {each_group}")
  df_pdo_search[each_group] = df_pdo_search['PROJ_DEV_OBJECTIVE_DESC'].apply(check_pfm_categories,category=each_group)

df_pdo_search_selected = df_pdo_search.set_index(['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC']).dropna(how='all').reset_index()

Processing category: School
Processing category: HGSF
Processing category: PIM
Processing category: WFP


D:\New User\Temp\ipykernel_12868\1428824184.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

D:\New User\Temp\ipykernel_12868\1428824184.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

D:\New User\Temp\ipykernel_12868\1428824184.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning

In [10]:
sel_projects_pdo = full_data[full_data['PROJ_ID'].isin(list(df_pdo_search_selected['PROJ_ID'].unique()))]

fig1_df = sel_projects_pdo.groupby(['PROJ_APPRVL_FY','PROJ_STAT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'PROJ_STAT_NAME':'Project Status'})
fig1 = px.bar(fig1_df,x='PROJ_APPRVL_FY',y='PROJ_ID',color='Project Status',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Set2)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year')
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1000,height=500)

In [21]:
fig1_df = sel_projects_pdo.groupby(['LEAD_GP_NAME','CNTRY_SHORT_NAME'])['PROJ_ID'].nunique().reset_index()
fig1_df = fig1_df.rename(columns={'CNTRY_SHORT_NAME':'Country','LEAD_GP_NAME':'Lead GP'})
fig1 = px.bar(fig1_df,x='Country',y='PROJ_ID',color='Lead GP',text='PROJ_ID', color_discrete_sequence=px.colors.qualitative.Safe)
fig1.update_layout(plot_bgcolor='white')
fig1.update_xaxes(title='Approval Fiscal Year', tickangle=10, tickfont=dict(size=8))
fig1.update_yaxes(title='Number of Projects')
fig1.update_layout(width=1500,height=400)